# Fair Code — Audit 01: COMPAS Criminal Justice Bias

> *A real algorithm used in US courtrooms flags Black defendants as high-risk at 87%. White defendants? 0.4%. Same system. Different outcomes.*

**Dataset:** ProPublica COMPAS dataset — `compas-scores-raw.csv`  
**Protected attribute:** Race  
**Proxy variable:** `CustodyStatus`  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

This notebook walks through the full Fair Code pipeline:
1. Load and explore the dataset
2. Train the biased model (race + proxy included)
3. Measure the fairness gap
4. Identify the proxy variable
5. Train the fair model (both removed)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import chi2_contingency

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT   = '#c9a84c'   # Fair Code gold
DANGER   = '#9b2335'   # red — bias
SAFE     = '#4a7c6f'   # teal — mitigated
MUTED    = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
# Load from the COMPAS folder (run this notebook from the repo root)
df_raw = pd.read_csv('COMPAS/compas-scores-raw.csv')
print(f'Raw dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# Filter to Black and white defendants — the comparison in the ProPublica analysis
df = df_raw[df_raw['race'].isin(['African-American', 'Caucasian'])].copy()

# Target: high-risk = DecileScore >= 5
df['high_risk'] = (df['DecileScore'] >= 5).astype(int)

print(f'Filtered dataset: {len(df):,} defendants')
print(f"\nRace breakdown:")
print(df['race'].value_counts().to_string())
print(f"\nOverall high-risk rate: {df['high_risk'].mean():.1%}")

In [ ]:
# Visualise the raw high-risk rate disparity before any model
raw_rates = df.groupby('race')['high_risk'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = [DANGER if r == 'African-American' else MUTED for r in raw_rates.index]
bars = ax.barh(raw_rates.index, raw_rates.values, color=colors, height=0.45)

for bar, val in zip(bars, raw_rates.values):
    ax.text(val + 0.8, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11, color=ACCENT)

ax.set_xlabel('High-risk flag rate (%)')
ax.set_title('Raw COMPAS high-risk rates by race (before any model)', color=ACCENT, pad=12)
ax.axvline(50, color=MUTED, linewidth=0.5, linestyle='--', alpha=0.4)
ax.set_xlim(0, 95)
plt.tight_layout()
plt.show()

gap = raw_rates['African-American'] - raw_rates['Caucasian']
print(f'\nRaw gap in dataset: {gap:.2f} percentage points')

## 3. Identify the Proxy Variable

A proxy variable correlates with a protected attribute even though it doesn't name it directly.
Removing `race` alone isn't enough — the model will reconstruct racial signal through correlated features.

**Candidate:** `CustodyStatus` — reflects over-policing of Black communities via pretrial detention patterns.

In [ ]:
def check_proxy(df, feature, protected_attr):
    """Chi-squared test of independence. p < 0.05 = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_attr])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'protected_attr': protected_attr,
            'p_value': round(p, 6), 'is_proxy': p < 0.05}

# Check CustodyStatus against race
result = check_proxy(df, 'CustodyStatus', 'race')
print(f"Feature:        {result['feature']}")
print(f"Protected attr: {result['protected_attr']}")
print(f"p-value:        {result['p_value']}")
print(f"Is proxy:       {result['is_proxy']}")

In [ ]:
# Cross-tabulation confirms the correlation
ct = pd.crosstab(df['CustodyStatus'], df['race'], normalize='columns').round(3)
print('CustodyStatus distribution by race (column proportions):')
print(ct.to_string())
print('\nConclusion: custody patterns differ significantly by race.')
print('Including CustodyStatus lets the model infer race even without the race column.')

## 4. Train the Biased Model

Features include `race` directly **and** `CustodyStatus` as a proxy.

In [ ]:
X_biased = pd.get_dummies(df[['race', 'Sex_Code_Text', 'CustodyStatus', 'MaritalStatus']])
y = df['high_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)

biased_preds = biased_model.predict(X_test)
biased_accuracy = accuracy_score(y_test, biased_preds)

results_biased = X_test.copy()
results_biased['pred'] = biased_preds
results_biased['race'] = df.loc[X_test.index, 'race'].values

rates_biased = results_biased.groupby('race')['pred'].mean() * 100
gap_biased = rates_biased['African-American'] - rates_biased['Caucasian']

print('--- BIASED MODEL RESULTS ---')
print()
print(f"Black Defendant High-Risk Rate: {rates_biased['African-American']:.2f}%")
print(f"White Defendant High-Risk Rate: {rates_biased['Caucasian']:.2f}%")
print()
print(f"Fairness Gap: {gap_biased:.2f}%")
print(f"Model Accuracy: {biased_accuracy:.2%}")

## 5. Train the Fair Model

`race` and `CustodyStatus` are both removed. Only non-proxy features remain.

In [ ]:
X_fair = pd.get_dummies(df[['Sex_Code_Text', 'MaritalStatus']])

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)

fair_preds = fair_model.predict(X_test_f)
fair_accuracy = accuracy_score(y_test_f, fair_preds)

results_fair = X_test_f.copy()
results_fair['pred'] = fair_preds
results_fair['race'] = df.loc[X_test_f.index, 'race'].values

rates_fair = results_fair.groupby('race')['pred'].mean() * 100
gap_fair = rates_fair['African-American'] - rates_fair['Caucasian']

print('--- MITIGATED (UNBIASED) RESULTS ---')
print()
print(f"Black Defendant High-Risk Rate: {rates_fair['African-American']:.2f}%")
print(f"White Defendant High-Risk Rate: {rates_fair['Caucasian']:.2f}%")
print()
print(f"New Fairness Gap: {gap_fair:.2f}%")
print(f"Model Accuracy: {fair_accuracy:.2%}")

## 6. Compare Results

In [ ]:
reduction = (gap_biased - gap_fair) / gap_biased * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('COMPAS — Biased vs Mitigated Model', color=ACCENT, fontsize=13, y=1.02)

groups = ['Black defendants', 'White defendants']
colors_b = [DANGER, MUTED]
colors_f = [SAFE, SAFE]

for ax, rates, colors, title in [
    (axes[0], rates_biased, colors_b, f'Biased model\nGap: {gap_biased:.2f}%'),
    (axes[1], rates_fair,   colors_f, f'Mitigated model\nGap: {gap_fair:.2f}%')
]:
    vals = [rates['African-American'], rates['Caucasian']]
    bars = ax.bar(groups, vals, color=colors, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1,
                f'{val:.1f}%', ha='center', fontsize=11, color=ACCENT)
    ax.set_ylim(0, 100)
    ax.set_ylabel('High-risk flag rate (%)')
    ax.set_title(title, color=MUTED, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Fairness gap before: {gap_biased:.2f}%')
print(f'Fairness gap after:  {gap_fair:.2f}%')
print(f'Reduction:           {reduction:.1f}%')

## Key Insight

Removing `race` alone is not enough. `CustodyStatus` carries the same racial signal because of historical over-policing patterns — Black defendants are disproportionately held in pretrial custody, so custody status correlates with race in the training data. The model learns this correlation and uses it to reconstruct race even when the column is absent.

**The fix:** Remove both the protected attribute **and** its proxy variables.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*